# Semantic Search

Documents and query are embedded separately for comparison.

Cosine similarity is computed between the query and each document.


In [1]:
from openai import OpenAI
import numpy as np
import pandas as pd

In [2]:
from kaggle_secrets import UserSecretsClient
secret_label = "OPENAI_API_KEY"
OPENAI_API_KEY = UserSecretsClient().get_secret(secret_label)
client = OpenAI(api_key=OPENAI_API_KEY)

In [3]:
response = client.embeddings.create(
    input="OpenAI embeddings convert text into vectors that represent meaning.",
    model="text-embedding-3-small"
)

embedding_vector = response.data[0].embedding

print(f"Model: text-embedding-3-small")
print(f"Embedding length: {len(embedding_vector)}")
print(f"First 5 dimensions: {embedding_vector[:5]}")

Model: text-embedding-3-small
Embedding length: 1536
First 5 dimensions: [-0.022318711504340172, -0.0015303738182410598, 0.01783829741179943, -0.008200199343264103, -0.004772161599248648]


In [4]:
def cosine_similarity(a, b):
    a, b = np.array(a), np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

In [5]:
text1 = "OpenAI provides advanced language models."
text2 = "OpenAI creates powerful AI models for text understanding."

emb1 = client.embeddings.create(
    input=text1, model="text-embedding-3-small").data[0].embedding
emb2 = client.embeddings.create(
    input=text2, model="text-embedding-3-small").data[0].embedding

similarity = cosine_similarity(emb1, emb2)
print(f"Cosine similarity: {similarity:.4f}")

Cosine similarity: 0.7509


In [6]:
documents = [
    "Python is widely used for data analysis and automation.",
    "JavaScript powers interactive web applications.",
    "Machine learning enables systems to learn from data automatically.",
    "C++ is known for high-performance computing."
]
query = "Which language is used to create web apps?"
emb2 = client.embeddings.create(input=query, model="text-embedding-3-small").data[0].embedding
score =[]
for i in documents :
    emb1 = client.embeddings.create(input=i, model="text-embedding-3-small").data[0].embedding
    similarity = cosine_similarity(emb1, emb2)
    score.append({"doc":i,"score":similarity})
resultat=pd.DataFrame(score)
print(resultat.sort_values(by = "score"))

                                                 doc     score
2  Machine learning enables systems to learn from...  0.088648
3       C++ is known for high-performance computing.  0.167440
0  Python is widely used for data analysis and au...  0.205678
1    JavaScript powers interactive web applications.  0.481264
